# ARLSTM Grid Sweep Multi-Model Evaluation

This notebook evaluates and compares **all 27 trained ARLSTM grid sweep models** stored locally in `model-runs/arlstm_grid_sweep/`.

### Hyperparameter Grid Axes Evaluated:
- **Learning Rate (`lr`)**: `0.0001`, `0.001`, `0.01`
- **Masking Fraction (`mf`)**: `0.1` (10%), `0.9` (90%), `0.99` (99%)
- **Hidden Size (`hs`)**: `128`, `256`, `512`
- **Random Seed (`seed`)**: `42`


In [1]:
%matplotlib inline
import os
import sys
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import xarray as xr
from IPython.display import display, HTML

# Resolve repository root
cwd = Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent, Path("/usr/local/google/home/kruparell/flood-forecasting")] if (p / "googlehydrology").is_dir() or (p / "tutorial").is_dir()),
    cwd
)
tut_dir = repo_root / "tutorial" if (repo_root / "tutorial").is_dir() else repo_root

for p in [str(repo_root), str(tut_dir), str(tut_dir / "scripts"), str(tut_dir / "notebooks")]:
    if p not in sys.path:
        sys.path.insert(0, p)

from googlehydrology.utils.config import Config
from googlehydrology.modelzoo.arlstm import ARLSTM
from googlehydrology.evaluation.tester import RegressionTester
from googlehydrology.datasetzoo.multimet import _convert_to_tensor
from googlehydrology.evaluation.metrics import calculate_metrics, AllNaNError

def compute_hydro_metrics(obs_np, sim_np):
    if np.all(np.isnan(obs_np)) or np.all(np.isnan(sim_np)):
        return {"NSE": np.nan, "KGE": np.nan, "Pearson-r": np.nan, "RMSE": np.nan}
    obs_da = xr.DataArray(obs_np, dims=["date"])
    sim_da = xr.DataArray(sim_np, dims=["date"])
    try:
        return calculate_metrics(obs_da, sim_da, metrics=["NSE", "KGE", "Pearson-r", "RMSE"])
    except AllNaNError:
        return {"NSE": np.nan, "KGE": np.nan, "Pearson-r": np.nan, "RMSE": np.nan}
    except Exception:
        return {"NSE": np.nan, "KGE": np.nan, "Pearson-r": np.nan, "RMSE": np.nan}

plt.rcParams["font.size"] = 11
plt.rcParams["figure.figsize"] = (12, 6)
print("Environment and GoogleHydrology modules initialized successfully.")

Environment and GoogleHydrology modules initialized successfully.


## 1. Multi-Model Grid Sweep Evaluation Loop

We iterate through all local model directories in `model-runs/arlstm_grid_sweep/`, load configurations, scalers, and PyTorch checkpoints, run inference for both **Full Streamflow Input** and **Zero Streamflow Input**, and aggregate performance metrics.


In [2]:
t0 = time.time()
grid_dir = repo_root / "model-runs" / "arlstm_grid_sweep"
model_folders = sorted([d for d in grid_dir.iterdir() if d.is_dir()])

# Fast Debug Settings: Set to None to evaluate all 27 models and all basins
num_debug_models = 3
num_debug_basins = 2

if num_debug_models is not None:
    model_folders = model_folders[:num_debug_models]

print(f"Evaluating {len(model_folders)} models in {grid_dir.name} across {num_debug_basins} basins for fast debugging...")

sweep_records = []
detailed_results = {}

for m_dir in model_folders:
    m_name = m_dir.name
    match = re.match(r"lr([\d\.]+)_mf([\d\.]+)_hs(\d+)_seed(\d+)", m_name)
    lr = float(match.group(1)) if match else None
    mf = float(match.group(2)) if match else None
    hs = int(match.group(3)) if match else None

    ar_cfg = Config(m_dir / "config.yml")
    ar_cfg._cfg["run_dir"] = m_dir
    ar_cfg._cfg["base_run_dir"] = m_dir
    if not Path(ar_cfg.statics_data_dir).exists():
        ar_cfg._cfg["statics_data_dir"] = Path("/usr/local/google/home/kruparell/Caravans_V2")
    if not Path(ar_cfg.dynamics_data_dir).exists():
        ar_cfg._cfg["dynamics_data_dir"] = Path("/usr/local/google/home/kruparell/Caravans_MultiMet")
    if not Path(ar_cfg.targets_data_dir).exists():
        ar_cfg._cfg["targets_data_dir"] = Path("/usr/local/google/home/kruparell/Caravans_V2")

    if (m_dir / "sampled_task0_200basins_seed42.txt").exists():
        ar_cfg._cfg["test_basin_file"] = m_dir / "sampled_task0_200basins_seed42.txt"
        ar_cfg._cfg["train_basin_file"] = m_dir / "sampled_task0_200basins_seed42.txt"
    elif not Path(ar_cfg.test_basin_file).exists():
        ar_cfg._cfg["test_basin_file"] = tut_dir / "basin-lists" / "10-basin-test.txt"
        ar_cfg._cfg["train_basin_file"] = tut_dir / "basin-lists" / "10-basin-train.txt"

    scaler_path = m_dir / "scaler.nc"
    if not scaler_path.exists():
        continue
    scaler_ds = xr.open_dataset(scaler_path)
    mu_q = float(scaler_ds["streamflow_sim"].sel(parameter="mean").values) if "streamflow_sim" in scaler_ds else 1.7772
    sigma_q = float(scaler_ds["streamflow_sim"].sel(parameter="std").values) if "streamflow_sim" in scaler_ds else 3.3810

    pt_files = sorted(list(m_dir.glob("model_epoch*.pt")))
    if not pt_files:
        continue
    latest_ckpt = pt_files[-1]

    tester = RegressionTester(cfg=ar_cfg, run_dir=m_dir, period="test", init_model=True)
    tester.model.eval()

    if num_debug_basins is not None:
        tester.basins = tester.basins[:num_debug_basins]

    zarr_path = m_dir / "test" / latest_ckpt.stem / "test_results.zarr"
    if zarr_path.exists():
        try:
            ds_check = xr.open_zarr(zarr_path, consolidated=False)
            _ = ds_check["streamflow_obs"].shape
        except Exception as e:
            print(f"Removing corrupted Zarr store at {zarr_path}: {e}")
            import shutil
            shutil.rmtree(zarr_path)

    if not zarr_path.exists():
        print(f"Executing fast evaluation for {m_name}...")
        tester.evaluate()
        zarr_path = sorted(list((m_dir / "test").glob("*/test_results.zarr")))[-1] if (m_dir / "test").exists() else None

    if zarr_path and zarr_path.exists():
        ds_zarr = xr.open_zarr(zarr_path, consolidated=False).compute()
        basins = [str(b) for b in ds_zarr["basin"].values]
        if num_debug_basins is not None:
            basins = basins[:num_debug_basins]
        dates = pd.to_datetime(ds_zarr["date"].values)

        nse_full_list, nse_zero_list = [], []
        kge_full_list, kge_zero_list = [], []
        predictions_dict = {}
        for b in basins:
            obs = ds_zarr["streamflow_obs"].sel(basin=b).isel(freq=0, time_step=0).values.flatten()
            sim_full = np.maximum(0.0, ds_zarr["streamflow_sim"].sel(basin=b).isel(freq=0, time_step=0).values.flatten())

            zero_val = (0.0 - mu_q) / sigma_q
            if b in tester.basins:
                idx = tester.basins.index(b)
                sample = tester.dataset[idx]
                batch_zero = tester.dataset.collate_fn([{k: _convert_to_tensor(k, v) for k, v in sample.items()}])
                if "x_d_hindcast" in batch_zero and "streamflow_shift1" in batch_zero["x_d_hindcast"]:
                    batch_zero["x_d_hindcast"]["streamflow_shift1"] = torch.full_like(batch_zero["x_d_hindcast"]["streamflow_shift1"], zero_val)
                elif "x_d" in batch_zero and "streamflow_shift1" in batch_zero["x_d"]:
                    batch_zero["x_d"]["streamflow_shift1"] = torch.full_like(batch_zero["x_d"]["streamflow_shift1"], zero_val)

                batch_zero = tester.model.pre_model_hook(batch_zero, is_train=False)
                with torch.no_grad():
                    out_zero = tester.model(batch_zero)
                    y_hat_zero = out_zero["y_hat"][0, :, 0].cpu().numpy()
                sim_zero_raw = np.maximum(0.0, y_hat_zero * sigma_q + mu_q)
                if len(sim_zero_raw) < len(obs):
                    sim_zero = np.full_like(obs, np.nan)
                    sim_zero[-len(sim_zero_raw):] = sim_zero_raw
                else:
                    sim_zero = sim_zero_raw[:len(obs)]
            else:
                sim_zero = np.full_like(obs, np.nan)

            m_full = compute_hydro_metrics(obs, sim_full)
            m_zero = compute_hydro_metrics(obs, sim_zero)

            nse_full_list.append(m_full["NSE"])
            nse_zero_list.append(m_zero["NSE"])
            kge_full_list.append(m_full["KGE"])
            kge_zero_list.append(m_zero["KGE"])
            predictions_dict[b] = {"dates": dates, "obs": obs, "sim_full": sim_full, "sim_zero": sim_zero}

        sweep_records.append({
            "Model": m_name,
            "Learning Rate": lr,
            "Masking Fraction": mf,
            "Hidden Size": hs,
            "Median NSE (Full)": np.median(nse_full_list),
            "Mean NSE (Full)": np.mean(nse_full_list),
            "Median KGE (Full)": np.median(kge_full_list),
            "Median NSE (Zero)": np.median(nse_zero_list),
            "Mean NSE (Zero)": np.mean(nse_zero_list),
            "Median KGE (Zero)": np.median(kge_zero_list),
            "NSE Degradation (Full-Zero)": np.median(nse_full_list) - np.median(nse_zero_list),
        })
        detailed_results[m_name] = {"nse_full": nse_full_list, "nse_zero": nse_zero_list, "predictions": predictions_dict}

df_sweep = pd.DataFrame(sweep_records).sort_values(by="Median NSE (Full)", ascending=False)
elapsed = time.time() - t0
print(f"Evaluated {len(df_sweep)} models in {elapsed:.2f}s.")
display(df_sweep.round(4))

Evaluating 3 models in arlstm_grid_sweep across 2 basins for fast debugging...
[ARLSTM Init] Base=MeanEmbeddingForecastLSTM, Cell Input Dim=42, Output Size=12
  [ARLSTM Forward Step t=000/180] NaNs=0/1 | Step Input Mean=-0.0056 | Pred Mean=0.0064
  [Sanity Check t=000] Masked Ratio=0/1 (0.0%) | x_ar Observed Mean=-0.2270, Std=0.0000
  [H1 Error Drift t=000] Mean Abs Error |y_hat - y| = 0.2472
  [ARLSTM Forward Step t=090/180] NaNs=0/1 | Step Input Mean=-0.0054 | Pred Mean=0.1092
  [Sanity Check t=090] Masked Ratio=0/1 (0.0%) | x_ar Observed Mean=-0.2270, Std=0.0000
  [H1 Error Drift t=090] Mean Abs Error |y_hat - y| = 0.0269
  [ARLSTM Forward Step t=179/180] NaNs=0/1 | Step Input Mean=-0.0053 | Pred Mean=0.0748
  [Sanity Check t=179] Masked Ratio=0/1 (0.0%) | x_ar Observed Mean=-0.2270, Std=0.0000
  [H1 Error Drift t=179] Mean Abs Error |y_hat - y| = 0.2449
  [ARLSTM Forward Step t=000/180] NaNs=0/1 | Step Input Mean=-0.0051 | Pred Mean=0.0065
  [Sanity Check t=000] Masked Ratio=0/1 (0

,Model,Learning Rate,Masking Fraction,Hidden Size,Median NSE (Full),Mean NSE (Full),Median KGE (Full),Median NSE (Zero),Mean NSE (Zero),Median KGE (Zero),NSE Degradation (Full-Zero)
1,lr0.0001_mf0.1_hs256_seed42,0.0001,0.1,256,0.2346,0.2346,0.1528,-4.5512,-4.5512,-4.6001,4.7858
0,lr0.0001_mf0.1_hs128_seed42,0.0001,0.1,128,0.1968,0.1968,0.1500,-9.4558,-9.4558,-6.6849,9.6526
2,lr0.0001_mf0.1_hs512_seed42,0.0001,0.1,512,0.1380,0.1380,-0.3738,-4.2671,-4.2671,-4.3549,4.4052


## 2. Hyperparameter Performance Heatmaps

Visualizing **Median NSE (Full Input)** across Learning Rate ($Y$-axis) and Masking Fraction ($X$-axis), faceted by Hidden Size.


In [3]:
hidden_sizes = sorted(df_sweep["Hidden Size"].unique())
fig, axes = plt.subplots(1, len(hidden_sizes), figsize=(18, 5), sharey=True)
fig.patch.set_facecolor("white")

for i, hs_val in enumerate(hidden_sizes):
    sub_df = df_sweep[df_sweep["Hidden Size"] == hs_val]
    pivot = sub_df.pivot(index="Learning Rate", columns="Masking Fraction", values="Median NSE (Full)")
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[i], cbar=(i == len(hidden_sizes)-1), vmin=0.0, vmax=1.0)
    axes[i].set_title(f"Hidden Size = {hs_val}", fontweight="bold")
    axes[i].set_xlabel("Masking Fraction", fontweight="bold")
    if i == 0:
        axes[i].set_ylabel("Learning Rate", fontweight="bold")

plt.suptitle("ARLSTM Grid Sweep: Median NSE (Full River Discharge Input)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


/tmp/ipykernel_509360/3534138539.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Robustness Analysis: Full vs. Zero Discharge Input

Analyzing how masking fraction during training impacts model reliance on autoregressive streamflow inputs during inference.


In [4]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("white")

sns.barplot(data=df_sweep, x="Masking Fraction", y="NSE Degradation (Full-Zero)", hue="Hidden Size", palette="viridis", ax=ax)
ax.set_title("NSE Performance Degradation when Autoregressive Input is Zeroed Out", fontweight="bold", fontsize=13)
ax.set_xlabel("Training Masking Fraction", fontweight="bold")
ax.set_ylabel("NSE Drop (Median NSE Full - Median NSE Zero)", fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


/tmp/ipykernel_509360/884884052.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Top Performing Models Summary


In [5]:
print("=== TOP 5 ARLSTM MODELS (RANKED BY MEDIAN NSE FULL INPUT) ===")
display(df_sweep.head(5)[["Model", "Learning Rate", "Masking Fraction", "Hidden Size", "Median NSE (Full)", "Median NSE (Zero)", "NSE Degradation (Full-Zero)"]].round(4))


=== TOP 5 ARLSTM MODELS (RANKED BY MEDIAN NSE FULL INPUT) ===


,Model,Learning Rate,Masking Fraction,Hidden Size,Median NSE (Full),Median NSE (Zero),NSE Degradation (Full-Zero)
1,lr0.0001_mf0.1_hs256_seed42,0.0001,0.1,256,0.2346,-4.5512,4.7858
0,lr0.0001_mf0.1_hs128_seed42,0.0001,0.1,128,0.1968,-9.4558,9.6526
2,lr0.0001_mf0.1_hs512_seed42,0.0001,0.1,512,0.1380,-4.2671,4.4052


## 5. Hydrograph Comparison: Top N Models

Function to plot hydrographs comparing the **observed streamflow** against the **top N models** with full river discharge input (left) and zero river discharge input (right).


In [ ]:
def plot_top_model_hydrographs(df_sweep, detailed_results, basin_id=None, top_n=5):
    """Plots hydrograph comparisons for top_n models with and without river discharge input.
    
    Parameters:
        df_sweep (pd.DataFrame): DataFrame containing aggregated model evaluation metrics.
        detailed_results (dict): Dictionary storing predictions per model and basin.
        basin_id (str, optional): Target basin ID to plot. Defaults to first available basin.
        top_n (int, optional): Number of top models to include in comparison (default 5).
    """
    n_models = min(top_n, len(df_sweep))
    top_full_models = df_sweep.sort_values(by="Median NSE (Full)", ascending=False).head(n_models)["Model"].tolist()
    top_zero_models = df_sweep.sort_values(by="Median NSE (Zero)", ascending=False).head(n_models)["Model"].tolist()

    first_model = top_full_models[0]
    available_basins = list(detailed_results[first_model]["predictions"].keys())
    if basin_id is None or basin_id not in available_basins:
        basin_id = available_basins[0]

    fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
    fig.patch.set_facecolor("white")
    colors = plt.cm.tab10(np.linspace(0, 1, n_models))

    # --- Left Subplot: Full Discharge Input ---
    ax_left = axes[0]
    sample_data = detailed_results[first_model]["predictions"][basin_id]
    dates = sample_data["dates"]
    obs = sample_data["obs"]

    ax_left.plot(dates, obs, color="black", linewidth=2.0, label="Observed Streamflow", linestyle="-")
    for idx, m_name in enumerate(top_full_models):
        pred = detailed_results[m_name]["predictions"][basin_id]
        ax_left.plot(dates, pred["sim_full"], label=f"{m_name}", color=colors[idx], linewidth=1.5, alpha=0.85)

    ax_left.set_title(f"Top {n_models} Models (Full River Discharge Input) - Basin {basin_id}", fontweight="bold", fontsize=12)
    ax_left.set_xlabel("Date", fontweight="bold")
    ax_left.set_ylabel("Discharge (m³/s)", fontweight="bold")
    ax_left.legend(loc="upper right", fontsize=8)
    ax_left.grid(True, linestyle="--", alpha=0.5)

    # --- Right Subplot: Zero Discharge Input ---
    ax_right = axes[1]
    ax_right.plot(dates, obs, color="black", linewidth=2.0, label="Observed Streamflow", linestyle="-")
    for idx, m_name in enumerate(top_zero_models):
        pred = detailed_results[m_name]["predictions"][basin_id]
        ax_right.plot(dates, pred["sim_zero"], label=f"{m_name}", color=colors[idx], linewidth=1.5, alpha=0.85)

    ax_right.set_title(f"Top {n_models} Models (Zero River Discharge Input) - Basin {basin_id}", fontweight="bold", fontsize=12)
    ax_right.set_xlabel("Date", fontweight="bold")
    ax_right.legend(loc="upper right", fontsize=8)
    ax_right.grid(True, linestyle="--", alpha=0.5)

    plt.suptitle(f"Hydrograph Comparison across Top {n_models} Models for Basin {basin_id}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

# Example Call: Plot Top 5 Models
plot_top_model_hydrographs(df_sweep, detailed_results, top_n=5)